# Kaggle 电影推荐竞赛 - 主工作流程

### 简介
本 Notebook 用于 Kaggle 电影推荐竞赛的开发和实验。
流程包括：
1.  **环境配置**: 设置常量和路径。
2.  **数据处理**: 加载数据并构建用户-物品交互矩阵 (URM)。
3.  **本地验证**: 将数据分割为训练集和验证集。
4.  **模型训练与评估**: 训练基线模型并评估其性能。
5.  **实验区域**: 用于测试和比较新模型。
6.  **生成提交**: 使用最佳模型在全量数据上训练并生成提交文件。# Kaggle 电影推荐竞赛 - 主工作流程

### 简介
本 Notebook 用于 Kaggle 电影推荐竞赛的开发和实验。
流程包括：
1.  **环境配置**: 设置常量和路径。
2.  **数据处理**: 加载数据并构建用户-物品交互矩阵 (URM)。
3.  **本地验证**: 将数据分割为训练集和验证集。
4.  **模型训练与评估**: 训练基线模型并评估其性能。
5.  **实验区域**: 用于测试和比较新模型。
6.  **生成提交**: 使用最佳模型在全量数据上训练并生成提交文件。# Kaggle 电影推荐竞赛 - 主工作流程

### 简介
本 Notebook 用于 Kaggle 电影推荐竞赛的开发和实验。
流程包括：
1.  **环境配置**: 设置常量和路径。
2.  **数据处理**: 加载数据并构建用户-物品交互矩阵 (URM)。
3.  **本地验证**: 将数据分割为训练集和验证集。
4.  **模型训练与评估**: 训练基线模型并评估其性能。
5.  **实验区域**: 用于测试和比较新模型。
6.  **生成提交**: 使用最佳模型在全量数据上训练并生成提交文件。

## 1. 导入必要的库

In [ ]:
# -*- coding: utf-8 -*-

import pandas as pd
import scipy.sparse as sps
import numpy as np
import os

from RecSys_Course_AT_PoliMi.Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from RecSys_Course_AT_PoliMi.Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from RecSys_Course_AT_PoliMi.Evaluation.Evaluator import EvaluatorHoldout
from RecSys_Course_AT_PoliMi.Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender

## 2. 项目配置与常量

In [ ]:
# 数据文件路径
DATA_FOLDER = 'dataset'
DATA_TRAIN_PATH = os.path.join(DATA_FOLDER, 'data_train.csv')
DATA_TARGET_USERS_PATH = os.path.join(DATA_FOLDER, 'data_target_users_test.csv')
SUBMISSION_PATH = 'submission.csv'

# 随机种子，用于确保实验结果的可复现性
RANDOM_SEED = 1234

# 本地验证时，训练集所占的百分比
TRAIN_PERCENTAGE = 0.80

# 评估时使用的推荐列表长度 (cutoff)
EVALUATION_CUTOFF = 20

# 设置全局随机种子
np.random.seed(RANDOM_SEED)

print("项目配置加载完成.")

## 3. 辅助函数
### 3.1 加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.

In [ ]:
def load_and_preprocess_data(file_path: str) -> sps.csr_matrix:
    """
    加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.
    """
    print("--- 正在加载和预处理数据... ---")
    df_train = pd.read_csv(file_path, dtype={'row': int, 'col': int})

    n_users = df_train['row'].max() + 1
    n_items = df_train['col'].max() + 1

    urm_all = sps.coo_matrix(
        ([1.0] * len(df_train['row']), (df_train['row'], df_train['col'])),
        shape=(n_users, n_items),
        dtype=float
    ).tocsr()
    print(f"数据加载完成. URM 维度: {urm_all.shape}")
    return urm_all


### 3.2 接收评估结果字典和模型名称，并以清晰的格式打印关键指标.

In [ ]:
def print_results_formatted(results_dict: dict, model_name: str = "Model"):
    """
    接收评估结果字典和模型名称，并以清晰的格式打印关键指标.
    """
    res = results_dict.get(EVALUATION_CUTOFF, {})

    print(f"--- 模型评估结果: {model_name} ---")
    print(f"--------------------------------------------------")
    print(f"{f'RECALL@{EVALUATION_CUTOFF}':<25}: {res.get('RECALL', -1):.4f}   <-- 竞赛官方指标")
    print(f"{f'PRECISION@{EVALUATION_CUTOFF}':<25}: {res.get('PRECISION', -1):.4f}")
    print(f"{f'MAP@{EVALUATION_CUTOFF}':<25}: {res.get('MAP', -1):.4f}")
    print(f"{f'HIT_RATE@{EVALUATION_CUTOFF}':<25}: {res.get('HIT_RATE', -1):.4f}")
    print(f"{f'ITEM_COVERAGE@{EVALUATION_CUTOFF}':<25}: {res.get('COVERAGE_ITEM', -1):.4f}")
    print(f"{f'AVG_POPULARITY@{EVALUATION_CUTOFF}':<25}: {res.get('AVERAGE_POPULARITY', -1):.4f}")
    print(f"--------------------------------------------------\n")

# 4. 本地验证流程
本节将加载数据，并将其分割为训练集和验证集，为模型评估做准备。

In [ ]:
# 加载数据
urm_all = load_and_preprocess_data(DATA_TRAIN_PATH)

# 分割数据用于本地验证
print("\n--- 正在分割数据用于本地验证... ---")
URM_train, URM_validation = split_train_in_two_percentage_global_sample(
    urm_all,
    train_percentage=TRAIN_PERCENTAGE
)
print("数据分割完成.")
print(f"训练集维度: {URM_train.shape}, 验证集维度: {URM_validation.shape}\n")


# 初始化评估器
evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[EVALUATION_CUTOFF])
print("评估器初始化完成.")

## 5. 模型训练与实验
在这里训练、评估和比较不同的推荐模型。

### 5.1 基线模型: Item-based KNN CF

In [ ]:
print("--- 正在训练基线模型 (ItemKNNCF)... ---")
# 初始化模型实例
recommender_itemknn = ItemKNNCFRecommender(URM_train)

# 训练模型 (拟合相似度矩阵)
# 这些是模型的超参数，是后续优化的重点
recommender_itemknn.fit(topK=100, shrink=10, similarity='cosine')
print("模型训练完成.\n")

# 评估模型
results_dict_itemknn, _ = evaluator_validation.evaluateRecommender(recommender_itemknn)

# 打印格式化的结果
print_results_formatted(results_dict_itemknn, "ItemKNNCF Baseline")

### 5.2 实验模型: P3alpha (Code)

In [ ]:
print("--- 正在训练实验模型 (P3alpha)... ---")
# 初始化模型实例
recommender_p3alpha = P3alphaRecommender(URM_train)

# 训练模型
# P3alpha 在隐式反馈数据集上通常表现优异
recommender_p3alpha.fit(topK=100, alpha=0.8, implicit=True)
print("模型训练完成.\n")

# 评估模型
results_dict_p3alpha, _ = evaluator_validation.evaluateRecommender(recommender_p3alpha)

# 打印格式化的结果
print_results_formatted(results_dict_p3alpha, "P3alpha")

## 6. 生成最终提交文件
当你通过本地验证找到了最佳模型和参数后，在这里使用 **全部训练数据** (`urm_all`) 进行训练，并为测试集用户生成推荐。

In [ ]:
# 1. 确定最终模型
# 假设我们通过实验发现 P3alpha 效果最好
final_model_class = P3alphaRecommender

# 2. 在 *全部* 数据上训练最终模型
print("--- 正在全量数据上训练最终模型... ---")
final_recommender = final_model_class(urm_all)
# 使用你通过超参数优化找到的最佳参数
final_recommender.fit(topK=100, alpha=0.8, implicit=True)
print("最终模型训练完成.\n")

# 3. 加载需要预测的目标用户
df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
target_user_ids = df_target_users['user_id'].values

# 4. 生成推荐并保存到文件
print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成推荐... ---")
submission = []
for user_id in target_user_ids:
    # 为每个用户推荐 N 个物品，并移除已交互过的
    recommended_items = final_recommender.recommend(
        user_id,
        cutoff=EVALUATION_CUTOFF,
        remove_seen_flag=True
    )

    # 将物品ID列表转换成空格分隔的字符串
    submission.append((user_id, ' '.join(map(str, recommended_items))))

# 5. 创建DataFrame并保存为CSV
df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
df_submission.to_csv(SUBMISSION_PATH, index=False)

print(f"提交文件 '{SUBMISSION_PATH}' 已成功生成!")
print(df_submission.head())